# Complete Standalone NIWT Pipeline

This notebook is fully self-contained with **zero external dependencies** beyond standard libraries.
All model architectures, loss functions, and utilities are inlined.

## Contents
1. Imports & Setup
2. Data Loading (PTB-XL)
3. Model Architectures (RickerBasisLayer, MasonRicker)
4. Loss Functions (MSE, STFT, BandPower, Composite)
5. Training Loop
6. Evaluation & Visualization

In [ ]:
# ============================================================
# SECTION 1: IMPORTS & SETUP
# ============================================================
import os, sys, warnings, time, json
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ============================================================
# SECTION 2: DATA LOADING
# ============================================================
class PTBXLDataset(Dataset):
    """PTB-XL ECG dataset loader."""
    def __init__(self, data_dir='/home/mithunmanivannan/data/ptbxl_tensors', 
                 split='train', input_leads=[0, 1, 7], seq_len=5000):
        self.data_dir = Path(data_dir)
        self.input_leads = input_leads
        self.seq_len = seq_len
        
        # Load file list
        self.files = sorted(list(self.data_dir.glob('*.pt')))
        
        # Split
        n_train = int(len(self.files) * 0.9)
        if split == 'train':
            self.files = self.files[:n_train]
        else:
            self.files = self.files[n_train:]
        
        print(f'Loaded {len(self.files)} files for {split}')
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        data = torch.load(self.files[idx])
        ecg = data['ecg']  # (12, seq_len)
        
        # Normalize to [0, 1]
        ecg_min = ecg.min()
        ecg_max = ecg.max()
        ecg = (ecg - ecg_min) / (ecg_max - ecg_min + 1e-8)
        
        # Pad/trim to seq_len
        if ecg.size(1) < self.seq_len:
            ecg = F.pad(ecg, (0, self.seq_len - ecg.size(1)))
        else:
            ecg = ecg[:, :self.seq_len]
        
        return {
            'input': ecg[self.input_leads, :],  # (3, seq_len)
            'target': ecg  # (12, seq_len)
        }

In [ ]:
# ============================================================
# SECTION 3: MODEL ARCHITECTURES
# ============================================================

class RickerBasisLayer(nn.Module):
    """
    Neural Inverse Wavelet Transform (NIWT) using Ricker wavelets.
    Predicts amplitude, position, and scale for K wavelets per output lead.
    """
    def __init__(self, input_dim, output_leads=12, seq_len=5000, num_wavelets=512,
                 sigma_min=0.02, sigma_max=0.5):
        super().__init__()
        self.output_leads = output_leads
        self.seq_len = seq_len
        self.num_wavelets = num_wavelets
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
        
        # Project latent to wavelet parameters (alpha, mu, sigma) for each wavelet
        self.proj = nn.Linear(input_dim, output_leads * num_wavelets * 3)
        
        # Time grid
        self.register_buffer('t_grid', torch.linspace(-1, 1, seq_len))
    
    def forward(self, x, chunk_size=32):
        B = x.size(0)
        params = self.proj(x).view(B, self.output_leads, self.num_wavelets, 3)
        t_grid = self.t_grid.view(1, 1, 1, self.seq_len)
        
        y = torch.zeros(B, self.output_leads, self.seq_len, device=x.device)
        
        for k_start in range(0, self.num_wavelets, chunk_size):
            k_end = min(k_start + chunk_size, self.num_wavelets)
            p = params[:, :, k_start:k_end, :]
            
            alpha = torch.clamp(p[..., 0], -10.0, 10.0).unsqueeze(-1)
            mu = torch.tanh(p[..., 1]).unsqueeze(-1)
            sigma = (self.sigma_min + (self.sigma_max - self.sigma_min) * 
                     torch.sigmoid(p[..., 2])).unsqueeze(-1)
            
            tau = torch.clamp((t_grid - mu) / sigma, -10.0, 10.0)
            psi = (1 - tau**2) * torch.exp(-0.5 * tau**2)
            y = y + torch.sum(alpha * psi, dim=2)
        
        return y


class ConvBlock(nn.Module):
    """Convolutional block with BatchNorm and GELU."""
    def __init__(self, in_ch, out_ch, kernel_size=7, stride=2):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, stride, padding=kernel_size//2)
        self.bn = nn.BatchNorm1d(out_ch)
        self.act = nn.GELU()
    
    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class MasonRicker(nn.Module):
    """
    Mason-style CNN encoder + Ricker wavelet decoder.
    Input: 3-lead ECG (I, II, V2)
    Output: 12-lead ECG reconstruction
    """
    def __init__(self, input_leads=3, hidden_dim=2048, num_wavelets=512,
                 encoder_channels=[64, 128, 256], sigma_min=0.02, sigma_max=0.5):
        super().__init__()
        
        # Encoder
        layers = [ConvBlock(input_leads, encoder_channels[0])]
        for i in range(1, len(encoder_channels)):
            layers.append(ConvBlock(encoder_channels[i-1], encoder_channels[i]))
        self.encoder = nn.Sequential(*layers)
        
        # Calculate encoder output size
        with torch.no_grad():
            dummy = torch.zeros(1, input_leads, 5000)
            enc_out = self.encoder(dummy)
            enc_dim = enc_out.view(1, -1).size(1)
        
        self.fc = nn.Linear(enc_dim, hidden_dim)
        self.ricker = RickerBasisLayer(hidden_dim, num_wavelets=num_wavelets,
                                       sigma_min=sigma_min, sigma_max=sigma_max)
    
    def forward(self, x):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = F.gelu(self.fc(z))
        return self.ricker(z)

In [ ]:
# ============================================================
# SECTION 4: LOSS FUNCTIONS
# ============================================================

class STFTLoss(nn.Module):
    """Multi-resolution STFT loss."""
    def __init__(self, fft_sizes=[512, 1024, 2048]):
        super().__init__()
        self.fft_sizes = fft_sizes
    
    def forward(self, pred, target):
        loss = 0
        for fft_size in self.fft_sizes:
            hop_size = fft_size // 4
            pred_stft = torch.stft(pred.view(-1, pred.size(-1)), n_fft=fft_size, 
                                   hop_length=hop_size, return_complex=True, center=True)
            target_stft = torch.stft(target.view(-1, target.size(-1)), n_fft=fft_size,
                                     hop_length=hop_size, return_complex=True, center=True)
            
            pred_mag = torch.abs(pred_stft)
            target_mag = torch.abs(target_stft)
            loss += F.l1_loss(pred_mag, target_mag)
            loss += F.l1_loss(torch.log(pred_mag + 1e-7), torch.log(target_mag + 1e-7))
        
        return loss / len(self.fft_sizes)


class BandPowerLoss(nn.Module):
    """Frequency band power matching."""
    def __init__(self, sr=500, bands=[(0, 5), (5, 40), (40, 100)]):
        super().__init__()
        self.sr = sr
        self.bands = bands
    
    def forward(self, pred, target):
        loss = 0
        n_fft = 1024
        
        pred_fft = torch.fft.rfft(pred.view(-1, pred.size(-1)), n=n_fft)
        target_fft = torch.fft.rfft(target.view(-1, target.size(-1)), n=n_fft)
        
        pred_psd = torch.abs(pred_fft) ** 2
        target_psd = torch.abs(target_fft) ** 2
        
        freqs = torch.fft.rfftfreq(n_fft, d=1/self.sr).to(pred.device)
        
        for low, high in self.bands:
            mask = (freqs >= low) & (freqs < high)
            loss += F.l1_loss(torch.log(pred_psd[:, mask].mean(1) + 1e-7),
                              torch.log(target_psd[:, mask].mean(1) + 1e-7))
        
        return loss / len(self.bands)


class CompositeLoss(nn.Module):
    """Combined MSE + STFT + BandPower loss."""
    def __init__(self, mse_w=1.0, stft_w=0.1, band_w=0.05):
        super().__init__()
        self.mse_w = mse_w
        self.stft_w = stft_w
        self.band_w = band_w
        self.stft = STFTLoss()
        self.band = BandPowerLoss()
    
    def forward(self, pred, target):
        return (self.mse_w * F.mse_loss(pred, target) + 
                self.stft_w * self.stft(pred, target) + 
                self.band_w * self.band(pred, target))

In [ ]:
# ============================================================
# SECTION 5: TRAINING LOOP
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='Training'):
        x = batch['input'].to(device)
        y = batch['target'].to(device)
        
        pred = model(x)
        loss = criterion(pred, y)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()
    total_rmse = 0
    total_pearson = 0
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating'):
            x = batch['input'].to(device)
            y = batch['target'].to(device)
            pred = model(x)
            
            # RMSE
            rmse = torch.sqrt(F.mse_loss(pred, y)).item()
            total_rmse += rmse
            
            # Pearson (simplified)
            p = pred.view(-1).cpu().numpy()
            g = y.view(-1).cpu().numpy()
            total_pearson += np.corrcoef(p, g)[0, 1]
    
    return {
        'rmse': total_rmse / len(loader),
        'pearson': total_pearson / len(loader)
    }

In [ ]:
# ============================================================
# SECTION 6: MAIN EXECUTION
# ============================================================

# Create datasets and loaders
train_ds = PTBXLDataset(split='train')
val_ds = PTBXLDataset(split='val')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Initialize model
model = MasonRicker(
    input_leads=3,
    hidden_dim=2048,
    num_wavelets=512,
    sigma_min=0.02,
    sigma_max=0.5
).to(device)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M')

# Setup training
criterion = CompositeLoss(mse_w=1.0, stft_w=0.1, band_w=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

In [ ]:
# Training loop
EPOCHS = 10  # Set to 50 for full training
best_rmse = float('inf')

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, device)
    scheduler.step()
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Val RMSE: {val_metrics['rmse']:.4f} | Val Pearson: {val_metrics['pearson']:.4f}")
    
    if val_metrics['rmse'] < best_rmse:
        best_rmse = val_metrics['rmse']
        torch.save(model.state_dict(), 'checkpoints/mason_ricker_best.pt')
        print('  -> Saved best model!')

In [ ]:
# ============================================================
# VISUALIZATION
# ============================================================

# Get a sample prediction
model.eval()
sample = val_ds[0]
x = sample['input'].unsqueeze(0).to(device)
y = sample['target'].unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(x)

# Plot
lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
fig, axes = plt.subplots(4, 3, figsize=(18, 12))

for i, (ax, name) in enumerate(zip(axes.flat, lead_names)):
    t = np.arange(1000) / 500
    ax.plot(t, y[0, i, :1000].cpu().numpy(), 'b-', alpha=0.7, label='GT')
    ax.plot(t, pred[0, i, :1000].cpu().numpy(), 'r-', alpha=0.7, label='Pred')
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('12-Lead ECG Reconstruction', fontweight='bold')
plt.tight_layout()
plt.show()